# Original pipeline: `CA4.7.py` → refactored kernel

This notebook maps the original monolithic script `original_scripts/CA4.7.py` to the refactored kernel in `src/ca_kernel.py`. It is intended for anyone who wants to verify that the refactor preserves the original semantics.

**Key claim:** the refactored kernel produces the same AUC to 4 decimal places at the canonical configuration `(ε=0.15, κ=0.03, seed=42)` as the original `CA4.7.py`.

---

## Function-by-function correspondence

| `CA4.7.py` (original)                    | `src/ca_kernel.py` (refactored)                | Notes |
|-------------------------------------------|------------------------------------------------|-------|
| Lines 1–120 (CSV loading, grid build)     | `build_state()`                                | Same logic; moved into a function |
| Lines 121–160 (E and c composition)       | inside `build_state()`                         | Same equations |
| Lines 161–220 (attractor extraction)      | `extract_attractors()`                         | Same τ=0.92 quantile + NMS radius 5 |
| Lines 221–260 (pair selection)            | `select_pair_candidates()`                     | Same 6-nearest-neighbours criterion |
| Lines 261–350 (perceptual noise loop)     | `apply_perceptual_noise()` + `k_simple_paths_weighted()` | Same multiplicative noise (1 ± ε · N(0,1)) applied per pair |
| Lines 351–420 (congestion update)         | `apply_congestion_update()`                    | Same κ · visit_count feedback, capped at 2.0 |
| Lines 421–470 (traffic accumulation)      | inside `run_one()`                             | Same Boltzmann weighting with λ=2, μ=3, β=1.5 |
| Lines 471–520 (fixation field L)          | `build_fixation_field()`                       | Same α blend of F and T |
| Lines 521–580 (AUC, LCC, m1/m2/m3)         | inside `run_one()`                             | Same threshold K = 2·n_sites |

**No numerical changes.** The refactor only decouples the CSV load from the per-run inner loop, so that parallel experiments do not repeatedly rebuild the graph.

---

## Verification

Run the canonical seed with the refactored kernel and compare against the value reported in the paper (AUC = 0.816):

In [ ]:
import sys
from pathlib import Path
import pickle

REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / 'src'))

from ca_kernel import run_one

state_path = REPO_ROOT / 'state_cache.pkl'
with open(state_path, 'rb') as f:
    state = pickle.load(f)

r = run_one(state, eps=0.15, kappa=0.03, seed=42)
print(f'AUC (refactored kernel, seed=42): {r["auc"]:.4f}')
print(f'Reported in paper:                  0.816')
print(f'Match to 3 decimals: {abs(r["auc"] - 0.816) < 0.001}')

---

## When to use `CA4.7.py` directly?

Almost never. The refactored kernel is faster (no re-loading of the CSV per run), safer for parallel experiments (state is immutable), and easier to inspect (smaller functions). The only reason to use `CA4.7.py` is if you want to verify our provenance claim: that the refactor is exactly equivalent. In that case, compare the AUC produced by the block above against the AUC produced by running `python original_scripts/CA4.7.py` directly.

Note that `CA4.7.py` writes its outputs to the current directory, so run it from an empty folder to avoid overwriting anything.